<a href="https://colab.research.google.com/github/Caroline-aprendendo/Agente_Financeiro/blob/main/Diario_Pessoal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
language="pt"

1.Gravando audio

In [2]:
import os
from datetime import datetime
from base64 import b64decode
from google.colab import output, drive
from IPython.display import display, Javascript, Audio

# Monta o Drive (será solicitado apenas uma vez)
drive.mount('/content/drive')

# Lógica em JavaScript para o microfone
RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
    const reader = new FileReader()
    reader.onloadend = e => resolve(e.srcElement.result)
    reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
    stream = await navigator.mediaDevices.getUserMedia({ audio: true })
    recorder = new MediaRecorder(stream)
    chunks = []
    recorder.ondataavailable = e => chunks.push(e.data)
    recorder.start()
    await sleep(time)
    recorder.onstop = async ()=>{
        blob = new Blob(chunks)
        text = await b2text(blob)
        resolve(text)
    }
    recorder.stop()
})
"""


Mounted at /content/drive


In [9]:
def gravar_audio_silencioso(segundos=5):
    """Versão da função que faz o trabalho sem imprimir mensagens"""
    local_path = "/content/temp_audio.wav"
    drive_folder = '/content/drive/MyDrive/MeuDiario'

    if not os.path.exists(drive_folder):
        os.makedirs(drive_folder)

    display(Javascript(RECORD))
    # O Python trava aqui até o JS terminar, garantindo o tempo certo
    js_result = output.eval_js(f'record({segundos * 1000})')

    audio = b64decode(js_result.split(',')[1])
    with open(local_path, 'wb') as f:
        f.write(audio)

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    drive_path = f"{drive_folder}/entrada_{timestamp}.wav"
    shutil.move(local_path, drive_path)
    return drive_path

def diario_com_botao_limpo(segundos=5):
    botao = widgets.Button(description='🎤 Gravar Diário', button_style='danger')
    saida = widgets.Output()

    def ao_clicar(b):
        with saida:
            # 1. Limpa o que estava escrito antes
            saida.clear_output()

            print(" [GRAVANDO] Pode falar agora...")

            # 2. Executa a gravação silenciosa
            caminho = gravar_audio_silencioso(segundos)

            # 3. Avisa que terminou
            print(f" [PARADO] Gravado com sucesso!")
            print(f" Arquivo: {caminho.split('/')[-1]}")

    botao.on_click(ao_clicar)
    display(botao, saida)

diario_com_botao_limpo(segundos=10)

Button(button_style='danger', description='🎤 Gravar Diário', style=ButtonStyle())

Output()

2.Transcrevendo o audio

In [11]:
! pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.3 MB/s eta 0:00:00


In [12]:
def transcrever_melhorado(record_file):
    import whisper
    # Carregando o modelo 'small' para mais precisão
    model = whisper.load_model("small")

    # Transcrevendo com as otimizações que você sugeriu
    result = model.transcribe(record_file, fp16=False, language=language)

    transcription = result["text"]
    print(f"\n📝 Diário: {transcription}")
    return transcription

# Chamada da função
texto_final = transcrever_melhorado(caminho_teste)

100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 92.3MiB/s]



📝 Diário:  Movie
